# Verification of Cell Density and Variation Results
This notebook verifies the values generated in `scores.csv` by recalculating the metrics directly from the raw `.dat` files.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

def read_dat_file(file_path):
    with open(file_path, 'r') as f:
        lines = f.read().splitlines()
        if not lines: return []
        count = int(lines[0].strip())
        values = [float(x.strip()) for x in lines[1:] if x.strip()]
        return values[:count]

def get_metrics_for_image(img_dir):
    all_areas = []
    a_files = sorted([f for f in img_dir.glob("A*.dat")], key=lambda x: int(x.stem[1:]) if x.stem[1:].isdigit() else 0)
    for f in a_files:
        all_areas.extend(read_dat_file(f))
    
    if not all_areas:
        return None
    
    arr = np.array(all_areas)
    mean_area = np.mean(arr)
    std_area = np.std(arr, ddof=0)
    
    return {
        'num': len(arr),
        'cd': 1e6 / mean_area,
        'cv': std_area / mean_area
    }

## 1. Load scores.csv and Calculate Raw Metrics

In [2]:
scores_path = Path("CD,CV,HEX-CAL - (CV as Weighted Distribution) - JM091826/result/scores.csv")
df_scores = pd.read_csv(scores_path)

data_root = Path("SAMPLE - DEKS-EYEBANK/Reader1")
dat_folder = next(p for p in data_root.iterdir() if p.is_dir() and p.name.startswith("DAT_"))

verification_data = []
for _, row in df_scores.iterrows():
    img_id = row['frame'].split('.')[0]
    img_dir = dat_folder / img_id
    
    if img_dir.exists():
        m = get_metrics_for_image(img_dir)
        if m:
            verification_data.append({
                'frame': row['frame'],
                'num_csv': row['num'],
                'num_calc': m['num'],
                'cd_csv': row['cd'],
                'cd_calc': m['cd'],
                'cv_csv': row['cv'],
                'cv_calc': m['cv']
            })

df_verify = pd.DataFrame(verification_data)
print(df_verify.head())

AttributeError: 'float' object has no attribute 'split'

## 2. Compare and Visualize Errors

In [ ]:
df_verify['num_diff'] = df_verify['num_csv'] - df_verify['num_calc']
df_verify['cd_diff'] = df_verify['cd_csv'] - df_verify['cd_calc']
df_verify['cv_diff'] = df_verify['cv_csv'] - df_verify['cv_calc']

print("Max Num Difference:", df_verify['num_diff'].abs().max())
print("Max CD Difference:", df_verify['cd_diff'].abs().max())
print("Max CV Difference:", df_verify['cv_diff'].abs().max())

plt.figure(figsize=(10, 6))
plt.plot(df_verify['frame'], df_verify['cd_diff'], marker='o', label='CD Error')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Frame')
plt.ylabel('Difference')
plt.title('CSV vs Calculation Error for CD')
plt.legend()
plt.show()